# Day 4 — Explore Wikipedia Articles Dataset

Quick EDA on `data/processed/wikipedia_articles.parquet` to verify
the ingestion worked correctly and understand what we're working with.

In [2]:
from pathlib import Path
import pyarrow.parquet as pq

PARQUET_PATH = Path("../data/processed/wikipedia_articles.parquet")
assert PARQUET_PATH.exists(), f"File not found: {PARQUET_PATH}"

table = pq.read_table(PARQUET_PATH)
print(f"Rows:    {table.num_rows:,}")
print(f"Columns: {table.column_names}")
print(f"Size:    {PARQUET_PATH.stat().st_size / 1e6:.1f} MB on disk")

Rows:    277,095
Columns: ['page_id', 'title', 'text', 'source', 'ingested_at', 'text_length']
Size:    170.7 MB on disk


## First 10 articles

In [3]:
df = table.to_pandas()
df.head(10)[["page_id", "title", "text_length", "source"]]

,page_id,title,text_length,source
0,1,April,16750,simplewiki-latest
1,2,August,10425,simplewiki-latest
2,6,Art,6176,simplewiki-latest
3,8,A,748,simplewiki-latest
4,9,Air,3007,simplewiki-latest
5,12,Autonomous communities of Spain,2067,simplewiki-latest
6,13,Alan Turing,2275,simplewiki-latest
7,14,Alanis Morissette,3761,simplewiki-latest
8,17,Adobe Illustrator,455,simplewiki-latest
9,18,Andouille,531,simplewiki-latest


## Basic statistics

In [4]:
print(f"Total articles:  {len(df):,}")
print(f"Total characters: {df['text_length'].sum():,}")
print()
print("Text length statistics (characters):")
print(df["text_length"].describe().to_string())

Total articles:  277,095
Total characters: 302,298,222

Text length statistics (characters):
count    277095.000000
mean       1090.955167
std        3835.503531
min          50.000000
25%         219.000000
50%         462.000000
75%        1015.000000
max      233955.000000


## Text length distribution

In [6]:
# Histogram of article lengths (capped at 20k chars for readability)
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Linear scale
    axes[0].hist(df["text_length"].clip(upper=20_000), bins=80, edgecolor="white")
    axes[0].set_xlabel("Text length (chars, capped at 20k)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Article length distribution")

    # Log scale
    axes[1].hist(df["text_length"], bins=80, edgecolor="white", log=True)
    axes[1].set_xlabel("Text length (chars)")
    axes[1].set_ylabel("Count (log)")
    axes[1].set_title("Article length distribution (log scale)")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — pip install matplotlib for plots")

matplotlib not installed — pip install matplotlib for plots


## Shortest and longest articles

Check the tails — are the shortest ones real content? Are the longest ones clean?

In [ ]:
print("=== 5 SHORTEST articles ===")
for _, row in df.nsmallest(5, "text_length").iterrows():
    print(f"\n--- {row['title']} (id={row['page_id']}, {row['text_length']} chars) ---")
    print(row["text"][:200])

In [ ]:
print("=== 5 LONGEST articles ===")
for _, row in df.nlargest(5, "text_length").iterrows():
    print(f"\n--- {row['title']} ({row['text_length']:,} chars) ---")
    print(row["text"][:300])
    print("...")

## Spot-check: read a specific article

Pick a well-known article and verify the text looks clean.

In [ ]:
# Search for an article by title
SEARCH_TITLE = "Earth"  # change this to whatever you want

matches = df[df["title"].str.contains(SEARCH_TITLE, case=False, na=False)]
print(f"Found {len(matches)} articles matching '{SEARCH_TITLE}':")
print(matches[["page_id", "title", "text_length"]].head(10).to_string(index=False))

if len(matches) > 0:
    # Show the first match
    best = matches.iloc[0]
    print(f"\n=== {best['title']} ({best['text_length']:,} chars) ===")
    print(best["text"][:1500])
    print("\n[...truncated...]" if best["text_length"] > 1500 else "")

## Check for leftover markup

Scan for common wiki markup that should have been stripped.

In [ ]:
# Sample 1000 articles and check for leftover markup
sample = df.sample(min(1000, len(df)), random_state=42)

checks = {
    "[[..]] wikilinks": r"\[\[",
    "{{..}} templates": r"\{\{",
    "<ref> tags": r"<ref",
    "''' bold": r"'''",
    "Category:": r"Category:",
    "{| table start": r"\{\|",
}

import re

print("Leftover markup check (in 1000-article sample):")
for label, pattern in checks.items():
    count = sample["text"].str.contains(pattern, regex=True, na=False).sum()
    pct = count / len(sample) * 100
    status = "ok" if pct < 5 else "REVIEW"
    print(f"  {label:25s} → {count:4d} articles ({pct:.1f}%) [{status}]")

## Check the manifest

In [ ]:
import yaml

manifest_path = Path("../data/processed/wikipedia_articles_manifest.yaml")
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = yaml.safe_load(f)
    for k, v in manifest.items():
        print(f"  {k}: {v}")
else:
    print("Manifest not found")

## Summary

If everything looks good:
- Reasonable article count (~270k+ for Simple English Wikipedia)
- Text is clean readable English, not wiki markup
- Shortest articles are real content (not stubs or redirects)
- Leftover markup percentages are low

Then Day 4 ingestion is complete. ✓